# 第 9 章　合成控制法
## Synthetic Control Method

::: {.callout-note}
## 本章要点

1. **适用场景**：单个处理单元 + 长时序，DID 和事件研究都难以应用时
2. **方法逻辑**：用控制单元的加权组合构造「合成反事实」
3. **权重估计**：最小化预处理期预测误差的二次规划
4. **推断**：安慰剂检验（Permutation Test）替代传统标准误
5. **Python 实现**：`pysyncon` 官方接口 + 手动 OLS 简化版
6. **与 DID 的关系**：同一套反事实逻辑，不同的控制组构造方式

**典型应用**：烟草税政策（Abadie & Gardeazabal, 2003）、
沪深港通开通对资本市场的影响、某省率先实施的监管政策。
:::


## 环境准备


In [ ]:
# ── 第 9 章　合成控制法　────────────────────────────────────────────────
import os, warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy import stats

warnings.filterwarnings('ignore')
matplotlib.rcParams['font.sans-serif'] = ['SimHei', 'Arial Unicode MS', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False
pd.set_option('display.float_format', '{:.4f}'.format)

OUTPUT = 'output'
os.makedirs(OUTPUT, exist_ok=True)
RNG = np.random.default_rng(42)
print('环境就绪 ✓')


---

## 0　引言：当只有一个「处理单元」时

第 8 章的 DID 需要足够多的处理单元来估计平均效应，
第 7 章的事件研究同样依赖多只股票来平均掉特异性噪声。

但很多重要的政策问题天然只有**一个处理单元**：
- 香港回归对香港经济的影响（Abadie et al., 2015）
- 沪深港通开通对 A 股市场的影响
- 某省率先实施碳交易机制对 GDP 的影响

对单个处理单元：
- 不能做 DID（没有足够的处理组内变动）
- 不能用事件研究（只有一只「股票」）
- 传统的「选几个对照省份取均值」存在任意性

**合成控制法（Abadie & Gardeazabal, 2003; Abadie et al., 2010）**
的核心思想：**让数据来决定控制单元的权重**，
找到一个加权平均，使得合成对照在预处理期的特征和结果
与处理单元尽可能吻合——这个合成对照就是处理单元的「反事实」。


---

## 1　方法逻辑

### 1.1　设置

- $J+1$ 个单元，单元 1 是处理单元，单元 $2, \ldots, J+1$ 是控制单元（捐赠池）
- 时间 $t = 1, \ldots, T_0, T_0+1, \ldots, T$，处理发生在 $T_0+1$
- 观测结果：$Y_{it}$，处理单元在处理后的反事实：$Y_{1t}(0)$（不可观测）

### 1.2　合成对照

寻找权重向量 $W = (w_2, \ldots, w_{J+1})$，$w_j \geq 0$，$\sum w_j = 1$，使得：

$$\sum_{t=1}^{T_0} \left( Y_{1t} - \sum_{j=2}^{J+1} w_j Y_{jt} \right)^2 \to \min$$

处理效应估计量：

$$\hat{\alpha}_{1t} = Y_{1t} - \sum_{j=2}^{J+1} \hat{w}_j Y_{jt}, \quad t > T_0$$

### 1.3　推断：安慰剂检验（In-Space Placebo）

合成控制法没有传统意义上的标准误——
预处理期只有一个处理单元的时序数据，无法估计抽样分布。

**替代方法**：对捐赠池中的每个控制单元分别做「假装它是处理单元」的合成控制，
得到 $J$ 条「安慰剂路径」。
如果真实处理单元的 Gap 远大于安慰剂 Gap 的分布，
就可以认为处理效应是显著的。

::: {.callout-important}
## 合成控制的识别假设

**无干扰假设（SUTVA）**：控制单元不受处理单元政策的影响。
如果「沪深港通」影响了整个市场，
那么「其他亚洲市场」就不是合格的捐赠池——
因为它们也可能受到溢出效应的影响。
:::


In [ ]:
# ── 2.1  构造演示数据 ────────────────────────────────────────────────────
# 场景：评估「沪深港通开通（2014 年 11 月）」对 A 股市场换手率的影响
# 处理单元：中国 A 股市场
# 捐赠池：6 个亚洲股票市场

YEARS     = list(range(2007, 2023))
TREAT_YR  = 2014
MARKETS   = ['China', 'Japan', 'Korea', 'HongKong',
             'Singapore', 'Taiwan', 'Thailand']
TRUE_EFFECT = 8.0    # 处理使换手率提高约 8 个百分点

np.random.seed(2024)

# 共同趋势（宏观因素）
base = np.linspace(50, 70, len(YEARS)) + np.random.normal(0, 3, len(YEARS))

# 各市场的数据（均受共同趋势影响，但有各自的特征）
market_data = {}
weights_true = [0, 0.45, 0.30, 0.15, 0.10, 0, 0]  # 真实合成权重（中国除外）

for i, mkt in enumerate(MARKETS):
    noise = np.random.normal(0, 4, len(YEARS))
    level = [55, 48, 52, 65, 58, 50, 45][i]
    series = base * ([1.2,0.9,1.0,1.3,0.95,0.85,0.80][i]) + noise + level - 50
    market_data[mkt] = series

# 在 China 的处理后加入处理效应
df_sc = pd.DataFrame(market_data, index=YEARS)
df_sc.index.name = 'year'
pre_mask  = np.array(YEARS) <= TREAT_YR
post_mask = ~pre_mask
for t_idx in range(len(YEARS)):
    if YEARS[t_idx] > TREAT_YR:
        yrs_after = YEARS[t_idx] - TREAT_YR
        df_sc.loc[YEARS[t_idx], 'China'] += TRUE_EFFECT * min(yrs_after/3, 1.2)

print('模拟亚洲市场换手率数据（%）：')
print(df_sc.round(1).to_string())
print(f'\n处理单元：China  处理年份：{TREAT_YR}  真实效应：+{TRUE_EFFECT}%')


---

## 3　Python 实现

### 3.1　方法一：手动二次规划（理解算法）


In [ ]:
# ── 3.1  手动合成控制：二次规划求权重 ───────────────────────────────────

def synthetic_control(df, treated_unit, donor_pool, pre_years, post_years):
    '''
    合成控制法核心实现。
    最小化预处理期内处理单元与合成对照的 MSE，
    约束：权重非负、权重和为 1。
    '''
    Y_treat = df.loc[pre_years, treated_unit].values
    Y_donor = df.loc[pre_years, donor_pool].values   # shape: (T0, J)

    J = len(donor_pool)

    # 目标函数：MSE
    def objective(w):
        synth = Y_donor @ w
        return np.mean((Y_treat - synth) ** 2)

    # 约束：非负 + 和为 1
    constraints = [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1}]
    bounds = [(0, 1)] * J
    w0     = np.ones(J) / J

    result = minimize(objective, w0, method='SLSQP',
                      bounds=bounds, constraints=constraints,
                      options={'ftol': 1e-12, 'maxiter': 1000})

    weights = result.x
    weights[weights < 1e-4] = 0   # 截断极小权重
    weights /= weights.sum()       # 重新归一化

    # 构造完整时序的合成对照
    all_years = pre_years + post_years
    Y_all_donor = df.loc[all_years, donor_pool].values
    Y_synth = Y_all_donor @ weights

    gap = df.loc[all_years, treated_unit].values - Y_synth

    return weights, Y_synth, gap, result.fun

# 运行合成控制
pre_years  = [y for y in YEARS if y <= TREAT_YR]
post_years = [y for y in YEARS if y >  TREAT_YR]
donor_pool = [m for m in MARKETS if m != 'China']

weights, Y_synth, gap, pre_mse = synthetic_control(
    df_sc, 'China', donor_pool, pre_years, post_years)

print('合成控制权重：')
for mkt, w in zip(donor_pool, weights):
    if w > 0.001:
        print(f'  {mkt:<12}: {w:.4f}')
print(f'\n预处理期 RMSE：{np.sqrt(pre_mse):.4f}%')
print(f'处理后平均 Gap：{gap[len(pre_years):].mean():.4f}%')
print(f'真实效应（均值）：约 {TRUE_EFFECT * 0.9:.1f}%（斜升式植入）')


In [ ]:
# ── 3.2  安慰剂检验（In-Space Placebo）────────────────────────────────

all_years_list = pre_years + post_years
placebo_gaps   = {}

for placebo_unit in donor_pool:
    placebo_donor = [m for m in MARKETS if m != placebo_unit]
    try:
        _, _, pg, pre_err = synthetic_control(
            df_sc, placebo_unit, placebo_donor, pre_years, post_years)
        # 只保留预处理期拟合较好的安慰剂（RMSE < 3 × 处理单元的 RMSE）
        if np.sqrt(pre_err) < 3 * np.sqrt(pre_mse):
            placebo_gaps[placebo_unit] = pg
    except Exception:
        pass

print(f'通过质量筛选的安慰剂单元：{list(placebo_gaps.keys())}')


In [ ]:
# ── 3.3  主图 + 安慰剂检验图 ─────────────────────────────────────────────

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# ── 左图：处理单元 vs 合成对照 ──────────────────────────────────────────
ax1.plot(all_years_list, df_sc.loc[all_years_list, 'China'].values,
         'steelblue', lw=2.5, marker='o', ms=5, label='中国 A 股（实际）')
ax1.plot(all_years_list, Y_synth,
         'darkorange', lw=2, linestyle='--', label='合成对照')
ax1.axvline(TREAT_YR + 0.5, color='red', lw=1.5, linestyle='--',
            label=f'沪深港通开通（{TREAT_YR}年底）')
ax1.axvspan(TREAT_YR + 0.5, max(all_years_list) + 0.5,
            alpha=0.06, color='orange')
ax1.set_title('合成控制：处理单元 vs 合成对照', fontsize=11)
ax1.set_xlabel('年份')
ax1.set_ylabel('市场换手率（%）')
ax1.legend(fontsize=9)

# ── 右图：Gap 图 + 安慰剂路径 ─────────────────────────────────────────
for unit, pg in placebo_gaps.items():
    ax2.plot(all_years_list, pg, color='gray', lw=0.8, alpha=0.5)

ax2.plot(all_years_list, gap, 'steelblue', lw=2.5, label='中国（处理效应）')
ax2.axhline(0, color='black', lw=0.8, linestyle=':')
ax2.axvline(TREAT_YR + 0.5, color='red', lw=1.5, linestyle='--')

# 安慰剂 90% 分位带
if placebo_gaps:
    pg_arr = np.array(list(placebo_gaps.values()))
    ax2.fill_between(all_years_list,
                     np.percentile(pg_arr, 5, axis=0),
                     np.percentile(pg_arr, 95, axis=0),
                     alpha=0.12, color='gray', label='安慰剂 90% 分位带')

ax2.set_title('Gap 图：处理效应 vs 安慰剂分布', fontsize=11)
ax2.set_xlabel('年份')
ax2.set_ylabel('Gap（处理单元 − 合成对照）')
ax2.legend(fontsize=9)

plt.suptitle('合成控制法：沪深港通对 A 股换手率的影响（模拟数据）',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig(f'{OUTPUT}/ch09_synth_control.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── 3.4  基于安慰剂的 p 值 ──────────────────────────────────────────────
# p 值 = 安慰剂的 post-treatment RMSPE / pre-treatment RMSPE 比值
# 超过处理单元比值的比例

def rmspe_ratio(gap_arr, n_pre):
    pre  = np.sqrt(np.mean(gap_arr[:n_pre]**2))
    post = np.sqrt(np.mean(gap_arr[n_pre:]**2))
    return post / (pre + 1e-10)

n_pre    = len(pre_years)
ratio_treat = rmspe_ratio(gap, n_pre)

ratios_placebo = []
for pg in placebo_gaps.values():
    ratios_placebo.append(rmspe_ratio(np.array(pg), n_pre))

if ratios_placebo:
    p_val = np.mean(np.array(ratios_placebo) >= ratio_treat)
    print(f'RMSPE 比值检验：')
    print(f'  处理单元比值：{ratio_treat:.3f}')
    print(f'  安慰剂比值均值：{np.mean(ratios_placebo):.3f}')
    print(f'  p 值（单侧）：{p_val:.4f}')
    sig = '显著（p < 0.1）' if p_val < 0.1 else '不显著'
    print(f'  结论：{sig}')


---

## 4　用 `pysyncon` 官方库


In [ ]:
# ── 4.1  pysyncon 接口 ───────────────────────────────────────────────────
try:
    from pysyncon import Dataprep, Synth
    PYSYNCON = True
    print('pysyncon 已安装 ✓')
except ImportError:
    PYSYNCON = False
    print('pysyncon 未安装（pip install pysyncon），跳过官方接口演示')
    print('手动实现（Section 3）的结果完全等价。')

if PYSYNCON:
    # 转为长表格式
    df_long = df_sc.reset_index().melt(
        id_vars='year', var_name='unit', value_name='turnover')

    dataprep = Dataprep(
        foo                   = df_long,
        predictors            = ['turnover'],
        predictors_op         = 'mean',
        time_predictors_prior = pre_years,
        special_predictors    = [],
        dependent             = 'turnover',
        unit_variable         = 'unit',
        time_variable         = 'year',
        treatment_identifier  = 'China',
        controls_identifier   = donor_pool,
        time_optimize_ssr     = pre_years,
        time_plot             = all_years_list,
    )
    synth = Synth()
    synth.fit(dataprep)

    print('\npysyncon 权重：')
    print(synth.W.round(4))

    synth.path_plot(time_period=all_years_list,
                    treatment_time=TREAT_YR + 1)
    plt.title('pysyncon：处理单元 vs 合成对照', fontsize=11)
    plt.tight_layout()
    plt.savefig(f'{OUTPUT}/ch09_pysyncon.png', dpi=150, bbox_inches='tight')
    plt.show()


---

## 5　合成控制 vs DID：场景选择指南

| 维度 | 合成控制 | DID |
|------|----------|-----|
| 处理单元数 | **1 个**（或极少数）| 多个 |
| 控制组构造 | 数据驱动的加权组合 | 直接对比 |
| 预处理期要求 | **长**（需要拟合，通常 ≥ 10 期）| 短（2 期即可）|
| 推断方法 | 安慰剂检验（置换检验）| 标准误 / Bootstrap |
| 平行趋势 | 通过预处理期拟合来近似满足 | 需要显式检验 |
| 典型数据频率 | 年度 / 季度 | 年度 / 月度 |
| 适合「渐进扩散」政策？| 否（需要明确处理时间）| 是（CS 估计量）|

**选择原则**：
- 处理单元 = 1 → 合成控制
- 处理单元多，政策分批推进 → DID + Callaway-Sant'Anna
- 二者都可用时 → 两种方法都做，作为互相验证的稳健性检验

::: {.callout-tip}
## 提示词模板：合成控制法

````
我有一个面板 DataFrame，列为 unit（地区/国家）、year、outcome（结果变量）。
处理单元是「unit_name」，处理发生在「treat_year」。

请帮我：
1. 用 scipy.optimize.minimize 实现合成控制权重估计
   （约束：权重非负，权重和为 1，最小化预处理期 MSE）
2. 绘制两张图：（a）处理单元 vs 合成对照路径图；
   （b）Gap 图，叠加所有安慰剂单元的 gap 路径
3. 计算 RMSPE 比值 p 值（post/pre RMSPE 超过处理单元的安慰剂比例）
4. 输出合成权重（只列出权重 > 0.01 的单元）
````
:::


---

## 6　章末练习

**练习 1（权重稳健性）**
改变捐赠池的组成（逐一排除某个控制单元），
观察合成权重如何变化，处理效应估计是否稳健。
这叫「Leave-one-out 安慰剂检验」。

**练习 2（预处理期长度）**
将预处理期从 8 年缩短为 4 年，比较合成权重和预处理期拟合质量的变化。
写一句话说明为什么合成控制法对预处理期长度比 DID 更敏感。

**练习 3（与 DID 对比）**
对同一份数据，做一个「简单 DID」：
把捐赠池中权重最大的控制单元作为对照组，
比较 DID 估计量和合成控制估计量的差异，讨论为什么不同。

**练习 4（思考题）**
某研究者想用合成控制法评估「2008 年北京奥运会」对北京 GDP 增速的影响。
请列出至少两个「捐赠池选择」的合理标准，
以及至少一个可能违反 SUTVA 假设的情形。
